<a href="https://colab.research.google.com/github/Techy-A/Stock-Market-Prediction-Analysis/blob/main/Stock_Market_BiLSTM_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stock Market Direction Prediction with ML and BiLSTM

Predicts whether a stock will close **up or down tomorrow**. The notebook downloads daily prices, builds about 75 technical features, trains Random Forest, XGBoost and a Bidirectional LSTM, and then asks the question that matters: **do any of them beat a trivial rule, after costs?**

**How the evaluation is kept honest**

| Rule | Why |
|:--|:--|
| Chronological train / validation / test split, never shuffled | A shuffled split lets the model see the future |
| Every feature is a ratio, rate or oscillator, never a raw price level | Price levels drift out of the training range, so trees learn "AAPL was below $150" instead of a pattern |
| Models are chosen on the validation block, the test block is touched once | Picking the winner by its test score inflates that score |
| Every model is compared with two baselines: "always up" and "same as today" | Stocks rise on most days, so "always up" already scores a high F1 |
| Balanced accuracy, MCC and ROC-AUC are reported next to F1 | F1 on the "up" class rewards predicting "up" every day |
| Each model's up/down cutoff is the median of its own validation probabilities | A fixed 0.5 cutoff fails when a model's probabilities are shifted, and the median needs no labels |
| The backtest charges a cost on every trade and runs a permutation test | A small edge has to survive costs and be distinguishable from luck |
| `SANITY_CHECK = True` swaps the prices for a random walk | On a random walk nothing is predictable, so any edge found there means a bug |

**Run it:** open in Colab, choose a T4 GPU (Runtime, Change runtime type), then Runtime, Run all. Change `TICKER` in cell 2 to try another stock.

In [ ]:
# ============================================================
#  CELL 1: Install and import
# ============================================================
!pip install yfinance ta xgboost shap --quiet

import os, random, warnings
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'          # hide TensorFlow's start-up log lines
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from scipy import stats

import yfinance as yf
import ta

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, matthews_corrcoef,
                             f1_score, precision_score, recall_score, roc_auc_score,
                             roc_curve, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Bidirectional, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import shap

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 60)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ============================================================
#  CELL 2: Configuration (the only cell you need to edit)
# ============================================================
TICKER       = 'AAPL'          # e.g. 'TSLA', 'MSFT', 'GOOGL', 'RELIANCE.NS', '^NSEI'
START_DATE   = '2019-01-01'
END_DATE     = datetime.today().strftime('%Y-%m-%d')
SANITY_CHECK = False           # True: replace prices with a random walk (no model should find an edge)

# Splits (chronological): the last TEST_FRAC is the test year,
# the last VAL_FRAC of what remains is the validation block used to choose models.
TEST_FRAC    = 0.15
VAL_FRAC     = 0.15
N_SPLITS     = 5               # TimeSeriesSplit folds inside the training block
SEED         = 42

# BiLSTM
SEQUENCE_LEN  = 60             # days of history per prediction
LSTM_UNITS    = 128
DROPOUT_RATE  = 0.3
LEARNING_RATE = 1e-3
BATCH_SIZE    = 64
MAX_EPOCHS    = 100

# Backtest
INITIAL_CAPITAL = 100_000
TRADING_DAYS    = 252
COST_BPS        = 5            # cost per position change in basis points (spread + commission)
RISK_FREE       = 0.04         # annual risk-free rate for the Sharpe ratio
N_PERMUTATIONS  = 2000         # permutation test size
WF_RETRAIN_DAYS = 21           # walk-forward: retrain the tree models every ~month

# Reproducibility: seeds Python, NumPy and TensorFlow in one call
tf.keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)

LABEL = 'Random walk (sanity check)' if SANITY_CHECK else TICKER      # used in chart titles
print(f"Ticker : {TICKER}{'  (SANITY CHECK: random-walk prices)' if SANITY_CHECK else ''}")
print(f"Period : {START_DATE} to {END_DATE}")
print(f"Costs  : {COST_BPS} bps per trade")

In [ ]:
# ============================================================
#  CELL 3: Download the data (or simulate it in sanity-check mode)
# ============================================================
def synthetic_ohlcv(start, end, seed=SEED, drift=0.0009, vol=0.018):
    """A geometric random walk with an upward drift similar to a large-cap stock.
    Tomorrow's move is independent of everything before it, so nothing is predictable."""
    r_ = np.random.default_rng(seed)
    idx = pd.bdate_range(start, end)
    close = 40 * np.exp(np.cumsum(r_.normal(drift, vol, len(idx))))
    open_ = close * np.exp(r_.normal(0, 0.006, len(idx)))
    high = np.maximum(open_, close) * np.exp(np.abs(r_.normal(0, 0.008, len(idx))))
    low = np.minimum(open_, close) * np.exp(-np.abs(r_.normal(0, 0.008, len(idx))))
    volume = r_.lognormal(18, 0.35, len(idx))
    return pd.DataFrame({'Open': open_, 'High': high, 'Low': low, 'Close': close, 'Volume': volume}, index=idx)

if SANITY_CHECK:
    raw_df = synthetic_ohlcv(START_DATE, END_DATE)
else:
    raw_df = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
    if isinstance(raw_df.columns, pd.MultiIndex):            # yfinance returns a MultiIndex for one ticker
        raw_df.columns = raw_df.columns.get_level_values(0)
    raw_df = raw_df[['Open', 'High', 'Low', 'Close', 'Volume']]

raw_df.index = pd.to_datetime(raw_df.index)
raw_df = raw_df.sort_index()
raw_df = raw_df[raw_df['Volume'] > 0]                          # drop non-trading rows
assert len(raw_df) > 500, f"Only {len(raw_df)} rows downloaded; check the ticker and dates."

print(f"Rows          : {len(raw_df):,}")
print(f"Date range    : {raw_df.index[0].date()} to {raw_df.index[-1].date()}")
print(f"Missing values: {raw_df.isnull().sum().sum()}")
display(raw_df.tail())

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(raw_df.index, raw_df['Close'], color='steelblue', lw=1.2, label='Close')
axes[0].fill_between(raw_df.index, raw_df['Low'], raw_df['High'], alpha=0.15, color='steelblue')
axes[0].set_title(f'{LABEL}: price history', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price'); axes[0].legend()
axes[1].bar(raw_df.index, raw_df['Volume'] / 1e6, color='coral', alpha=0.7, width=1)
axes[1].set_ylabel('Volume (M)'); axes[1].set_title('Volume')
plt.tight_layout(); plt.savefig('price_history.png', dpi=150, bbox_inches='tight'); plt.show()

## Feature engineering

Every feature is scale-free: a return, a ratio to a moving average, an oscillator or a normalised width. Raw price levels (moving averages, band edges, ATR in dollars, cumulative On-Balance Volume) are deliberately left out. Their values in the test year sit outside anything seen in training, and tree models then split on "price above X", which is a fact about the past rather than a pattern.

The target for day *t* is 1 if the close on day *t+1* is above the close on day *t*. Features for day *t* only use data up to the close of day *t*.

In [ ]:
# ============================================================
#  CELL 4: Feature engineering (stationary features only)
# ============================================================
def engineer_features(raw: pd.DataFrame) -> pd.DataFrame:
    close, high, low, open_, volume = raw['Close'], raw['High'], raw['Low'], raw['Open'], raw['Volume']
    d = pd.DataFrame(index=raw.index)

    # 1. Returns and intraday shape
    for h in [1, 2, 5, 10, 21]:
        d[f'return_{h}d'] = close.pct_change(h)
    d['log_return_1d'] = np.log(close / close.shift(1))
    d['hl_ratio']      = (high - low) / close                       # intraday range
    d['co_ratio']      = (close - open_) / open_                    # open-to-close drift
    d['gap']           = open_ / close.shift(1) - 1                 # overnight gap

    # 2. Trend: distance from moving averages, crossovers, MACD (scaled by price), ADX, PSAR
    sma = {w: close.rolling(w).mean() for w in [5, 10, 20, 50, 100, 200]}
    for w in sma:
        d[f'close_vs_sma{w}'] = close / sma[w] - 1
        d[f'close_vs_ema{w}'] = close / close.ewm(span=w, adjust=False).mean() - 1
    d['sma_cross_5_20']   = sma[5] / sma[20] - 1
    d['sma_cross_50_200'] = sma[50] / sma[200] - 1
    macd = ta.trend.MACD(close)
    d['macd']        = macd.macd() / close
    d['macd_signal'] = macd.macd_signal() / close
    d['macd_diff']   = macd.macd_diff() / close
    adx = ta.trend.ADXIndicator(high, low, close)
    d['adx'], d['adx_pos'], d['adx_neg'] = adx.adx(), adx.adx_pos(), adx.adx_neg()
    d['close_vs_psar'] = close / ta.trend.PSARIndicator(high, low, close).psar() - 1

    # 3. Momentum oscillators
    d['rsi_14'] = ta.momentum.RSIIndicator(close, window=14).rsi()
    d['rsi_28'] = ta.momentum.RSIIndicator(close, window=28).rsi()
    d['rsi_overbought'] = (d['rsi_14'] > 70).astype(int)
    d['rsi_oversold']   = (d['rsi_14'] < 30).astype(int)
    stoch = ta.momentum.StochasticOscillator(high, low, close)
    d['stoch_k'], d['stoch_d'] = stoch.stoch(), stoch.stoch_signal()
    d['williams_r'] = ta.momentum.WilliamsRIndicator(high, low, close).williams_r()
    d['roc_5']  = ta.momentum.ROCIndicator(close, window=5).roc()
    d['roc_21'] = ta.momentum.ROCIndicator(close, window=21).roc()
    d['cci'] = ta.trend.CCIIndicator(high, low, close).cci()
    d['mfi'] = ta.volume.MFIIndicator(high, low, close, volume).money_flow_index()

    # 4. Volatility (all relative to price)
    bb = ta.volatility.BollingerBands(close, window=20, window_dev=2)
    d['bb_width'], d['bb_pct'] = bb.bollinger_wband(), bb.bollinger_pband()
    d['atr_pct'] = ta.volatility.AverageTrueRange(high, low, close, window=14).average_true_range() / close
    d['kc_pct']  = ta.volatility.KeltnerChannel(high, low, close).keltner_channel_pband()
    dc = ta.volatility.DonchianChannel(high, low, close, window=20)
    dc_hi, dc_lo = dc.donchian_channel_hband(), dc.donchian_channel_lband()
    d['dc_pct']   = (close - dc_lo) / (dc_hi - dc_lo)
    d['dc_width'] = (dc_hi - dc_lo) / close
    d['vol_5']  = d['log_return_1d'].rolling(5).std()
    d['vol_21'] = d['log_return_1d'].rolling(21).std()
    d['vol_ratio'] = d['vol_5'] / d['vol_21']

    # 5. Volume (relative, not absolute)
    d['volume_ratio'] = volume / volume.rolling(20).mean()
    obv = ta.volume.OnBalanceVolumeIndicator(close, volume).on_balance_volume()
    d['obv_flow_21']  = obv.diff(21) / volume.rolling(21).sum()     # net share of "up-day" volume, -1 to 1
    vwap_20 = (close * volume).rolling(20).sum() / volume.rolling(20).sum()
    d['close_vs_vwap20'] = close / vwap_20 - 1

    # 6. Lags and rolling statistics
    for lag in [1, 2, 3, 5, 10]:
        d[f'lag_return_{lag}'] = d['return_1d'].shift(lag)
        d[f'lag_volume_{lag}'] = d['volume_ratio'].shift(lag)
        d[f'lag_rsi_{lag}']    = d['rsi_14'].shift(lag)
    d['skew_21']   = d['log_return_1d'].rolling(21).skew()
    d['kurt_21']   = d['log_return_1d'].rolling(21).kurt()
    d['corr_5_21'] = d['return_5d'].rolling(21).corr(d['return_1d'])
    return d.replace([np.inf, -np.inf], np.nan)

features = engineer_features(raw_df)
FEATURE_COLS = list(features.columns)

close = raw_df['Close']
next_return = close.shift(-1) / close - 1                     # the return the strategy would earn by holding
labels = pd.DataFrame({'target': (next_return > 0).astype(int), 'next_return': next_return, 'Close': close})

features_ok = features.dropna()                               # rows where every feature exists
live_date = features_ok.index[-1]                             # latest day: features known, outcome not yet
df = features_ok.join(labels).dropna(subset=['next_return'])  # rows with a known outcome, used for modelling

print(f"Features          : {len(FEATURE_COLS)}")
print(f"Modelling rows    : {len(df):,}  ({df.index[0].date()} to {df.index[-1].date()})")
print(f"Latest row (live) : {live_date.date()}")
print(f"Up days / down days: {df['target'].sum()} / {(1 - df['target']).sum()}  "
      f"({df['target'].mean():.1%} up)")

In [ ]:
# ============================================================
#  CELL 5: Exploratory analysis
# ============================================================
key_features = ['rsi_14', 'macd_diff', 'bb_pct', 'atr_pct', 'return_1d',
                'vol_21', 'volume_ratio', 'adx', 'stoch_k']
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle(f'{LABEL}: feature distributions on up vs down days', fontsize=15, fontweight='bold', y=1.01)
for ax, feat in zip(axes.flat, key_features):
    for label, color in [(0, '#e74c3c'), (1, '#2ecc71')]:
        ax.hist(df.loc[df['target'] == label, feat], bins=40, alpha=0.55, color=color,
                label='Down' if label == 0 else 'Up', density=True)
    ax.set_title(feat, fontsize=10, fontweight='bold'); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight'); plt.show()

top_features = ['return_1d', 'rsi_14', 'macd_diff', 'bb_pct', 'atr_pct',
                'vol_21', 'volume_ratio', 'adx', 'roc_5', 'target']
fig, ax = plt.subplots(figsize=(11, 9))
corr = df[top_features].corr()
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Feature correlations (note how close to 0 the target row is)', fontweight='bold')
plt.tight_layout(); plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight'); plt.show()

## Train, validation and test

| Block | Used for |
|:--|:--|
| **Train** | Fitting every model and time-series cross-validation. Its last 15% is held back for early stopping (XGBoost trees, BiLSTM epochs) |
| **Validation** | Choosing the best model, and nothing else |
| **Test** (the most recent ~15%) | One final, untouched evaluation and the backtest |

All three models are trained on the same rows and predict exactly the same test days, so the comparison is fair. The BiLSTM's 60-day windows may reach back into the previous block for *features* only, which is information that was available at the time.

In [ ]:
# ============================================================
#  CELL 6: Chronological split and the evaluation helpers
# ============================================================
n = len(df)
n_test = int(n * TEST_FRAC)
n_dev = n - n_test
n_val = int(n_dev * VAL_FRAC)
n_train = n_dev - n_val
TRAIN, VAL, TEST = np.arange(0, n_train), np.arange(n_train, n_dev), np.arange(n_dev, n)
# Early stopping gets its own slice (the end of TRAIN), so VAL stays clean for choosing the best model
n_inner = int(n_train * 0.15)
FIT, INNER = TRAIN[:-n_inner], TRAIN[-n_inner:]

X_raw = df[FEATURE_COLS].values                    # trees are scale-invariant, so they use raw features
y = df['target'].values.astype(int)
r_next = df['next_return'].values
dates = df.index

scaler = RobustScaler().fit(X_raw[TRAIN])          # fitted on training rows only
X_scaled = scaler.transform(X_raw)                 # for the BiLSTM

split_info = pd.DataFrame({
    'Rows': [len(TRAIN), len(VAL), len(TEST)],
    'From': [dates[TRAIN[0]].date(), dates[VAL[0]].date(), dates[TEST[0]].date()],
    'To':   [dates[TRAIN[-1]].date(), dates[VAL[-1]].date(), dates[TEST[-1]].date()],
    'Up %': [y[TRAIN].mean(), y[VAL].mean(), y[TEST].mean()],
}, index=['Train', 'Validation', 'Test'])
display(split_info)

def evaluate(y_true, prob, threshold=0.5):
    """Direction-classification metrics. F1 is kept for reference, but it rewards
    predicting 'up' every day, so balanced accuracy and MCC are the ones to read."""
    prob = np.asarray(prob, dtype=float)
    pred = (prob > threshold).astype(int)
    auc = roc_auc_score(y_true, prob) if len(np.unique(prob)) > 1 else 0.5
    return {
        'Accuracy':      accuracy_score(y_true, pred),
        'Balanced acc':  balanced_accuracy_score(y_true, pred),
        'MCC':           matthews_corrcoef(y_true, pred) if len(np.unique(pred)) > 1 else 0.0,
        'ROC-AUC':       auc,
        'F1 (up)':       f1_score(y_true, pred, zero_division=0),
        'Predicted up':  pred.mean(),
    }

# Baselines that need no model at all
prob_always_up  = np.ones(n)                                   # "the stock goes up tomorrow"
prob_same_as_today = (df['return_1d'].values > 0).astype(float)  # "tomorrow repeats today"
baseline_probs = {'Baseline: always up': prob_always_up, 'Baseline: same as today': prob_same_as_today}
display(pd.DataFrame({k: evaluate(y[TEST], v[TEST]) for k, v in baseline_probs.items()}).T)

In [ ]:
# ============================================================
#  CELL 7: Random Forest
# ============================================================
rf_params = dict(n_estimators=500, max_depth=8, min_samples_split=20, min_samples_leaf=10,
                 max_features='sqrt', class_weight='balanced', random_state=SEED, n_jobs=-1)

# Time-series cross-validation inside the training block (for reporting stability)
rf_cv = []
for fold, (tr, va) in enumerate(TimeSeriesSplit(n_splits=N_SPLITS).split(TRAIN)):
    m = RandomForestClassifier(**rf_params).fit(X_raw[TRAIN][tr], y[TRAIN][tr])
    rf_cv.append(evaluate(y[TRAIN][va], m.predict_proba(X_raw[TRAIN][va])[:, 1]))
rf_cv = pd.DataFrame(rf_cv, index=[f'Fold {i+1}' for i in range(N_SPLITS)])
display(rf_cv[['Balanced acc', 'MCC', 'ROC-AUC', 'Predicted up']])

rf_model = RandomForestClassifier(**rf_params).fit(X_raw[TRAIN], y[TRAIN])
rf_prob = rf_model.predict_proba(X_raw)[:, 1]                  # probabilities for every row

# Decision threshold: the median of the model's own validation probabilities (uses no labels).
# A fixed 0.5 fails when a model's probabilities are shifted, e.g. all between 0.40 and 0.48.
THRESHOLD = {'Random Forest': float(np.median(rf_prob[VAL]))}
display(pd.DataFrame({'Validation': evaluate(y[VAL], rf_prob[VAL], THRESHOLD['Random Forest']),
                      'Test': evaluate(y[TEST], rf_prob[TEST], THRESHOLD['Random Forest'])}).T)

feat_imp = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.head(20).plot(kind='barh', ax=ax, color='steelblue'); ax.invert_yaxis()
ax.set_title('Random Forest: top 20 feature importances', fontweight='bold')
plt.tight_layout(); plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
#  CELL 8: XGBoost (early stopping on a slice of the training block)
# ============================================================
pos_weight = (1 - y[TRAIN].mean()) / y[TRAIN].mean()
xgb_params = dict(max_depth=5, learning_rate=0.05, subsample=0.8, colsample_bytree=0.7,
                  min_child_weight=10, reg_alpha=0.1, reg_lambda=1.0, scale_pos_weight=pos_weight,
                  eval_metric='logloss', random_state=SEED, n_jobs=-1, verbosity=0)

xgb_cv = []
for fold, (tr, va) in enumerate(TimeSeriesSplit(n_splits=N_SPLITS).split(TRAIN)):
    m = XGBClassifier(n_estimators=300, **xgb_params).fit(X_raw[TRAIN][tr], y[TRAIN][tr])
    xgb_cv.append(evaluate(y[TRAIN][va], m.predict_proba(X_raw[TRAIN][va])[:, 1]))
xgb_cv = pd.DataFrame(xgb_cv, index=[f'Fold {i+1}' for i in range(N_SPLITS)])
display(xgb_cv[['Balanced acc', 'MCC', 'ROC-AUC', 'Predicted up']])

# Choose the number of trees on the inner early-stopping slice, then refit on all training rows
es = XGBClassifier(n_estimators=1000, early_stopping_rounds=50, **xgb_params)
es.fit(X_raw[FIT], y[FIT], eval_set=[(X_raw[INNER], y[INNER])], verbose=False)
XGB_TREES = max(es.best_iteration + 1, 20)          # a floor keeps the model from collapsing to a stump
print(f"Early stopping chose {es.best_iteration + 1} trees (using {XGB_TREES})")
xgb_model = XGBClassifier(n_estimators=XGB_TREES, **xgb_params).fit(X_raw[TRAIN], y[TRAIN])
xgb_prob = xgb_model.predict_proba(X_raw)[:, 1]

THRESHOLD['XGBoost'] = float(np.median(xgb_prob[VAL]))
display(pd.DataFrame({'Validation': evaluate(y[VAL], xgb_prob[VAL], THRESHOLD['XGBoost']),
                      'Test': evaluate(y[TEST], xgb_prob[TEST], THRESHOLD['XGBoost'])}).T)

In [ ]:
# ============================================================
#  CELL 9: Bidirectional LSTM
# ============================================================
def windows_ending_at(X_arr, idx, seq_len):
    """One window per index t, covering days t-seq_len+1 .. t (inclusive).
    The window ends on day t because day t's features are known when predicting day t+1."""
    return np.stack([X_arr[t - seq_len + 1 : t + 1] for t in idx])

seq_fit_idx = FIT[FIT >= SEQUENCE_LEN - 1]                 # the first rows lack a full history
Xs_fit,   ys_fit   = windows_ending_at(X_scaled, seq_fit_idx, SEQUENCE_LEN), y[seq_fit_idx]
Xs_inner, ys_inner = windows_ending_at(X_scaled, INNER, SEQUENCE_LEN), y[INNER]   # early stopping only
Xs_val             = windows_ending_at(X_scaled, VAL, SEQUENCE_LEN)
Xs_test            = windows_ending_at(X_scaled, TEST, SEQUENCE_LEN)
print(f"Sequences  fit {Xs_fit.shape}  early-stop {Xs_inner.shape}  validation {Xs_val.shape}  test {Xs_test.shape}")

def build_bilstm(seq_len, n_features):
    model = Sequential([
        Input(shape=(seq_len, n_features)),
        Bidirectional(LSTM(LSTM_UNITS, return_sequences=True)), BatchNormalization(), Dropout(DROPOUT_RATE),
        Bidirectional(LSTM(LSTM_UNITS // 2)), BatchNormalization(), Dropout(DROPOUT_RATE),
        Dense(32, activation='relu'), Dropout(DROPOUT_RATE / 2),
        Dense(1, activation='sigmoid'),
    ])
    model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy',
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    return model

bilstm = build_bilstm(SEQUENCE_LEN, len(FEATURE_COLS))
weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=ys_fit)
history = bilstm.fit(
    Xs_fit, ys_fit, validation_data=(Xs_inner, ys_inner),
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE, class_weight={0: weights[0], 1: weights[1]},
    callbacks=[EarlyStopping(monitor='val_auc', mode='max', patience=12, restore_best_weights=True),
               ReduceLROnPlateau(monitor='val_loss', patience=6, factor=0.5, min_lr=1e-6)],
    verbose=0)
print(f"Trained for {len(history.history['loss'])} epochs "
      f"(best early-stopping AUC {max(history.history['val_auc']):.3f})")

bilstm_prob = np.full(n, np.nan)
bilstm_prob[VAL]  = bilstm.predict(Xs_val, verbose=0).ravel()
bilstm_prob[TEST] = bilstm.predict(Xs_test, verbose=0).ravel()

THRESHOLD['BiLSTM'] = float(np.median(bilstm_prob[VAL]))
display(pd.DataFrame({'Validation': evaluate(y[VAL], bilstm_prob[VAL], THRESHOLD['BiLSTM']),
                      'Test': evaluate(y[TEST], bilstm_prob[TEST], THRESHOLD['BiLSTM'])}).T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, key, title in [(axes[0], 'loss', 'Loss'), (axes[1], 'auc', 'AUC')]:
    ax.plot(history.history[key], label=f'Train {title}', color='steelblue')
    ax.plot(history.history[f'val_{key}'], label=f'Early-stopping slice {title}', color='coral')
    ax.set_title(f'BiLSTM: {title}', fontweight='bold'); ax.legend()
plt.tight_layout(); plt.savefig('bilstm_training_history.png', dpi=150, bbox_inches='tight'); plt.show()

## Model comparison

The best model is chosen by **validation** balanced accuracy, then all models are reported on the test block next to the baselines. The question to ask of the test table is not "which F1 is highest" but "does any model beat *always up* on balanced accuracy, MCC and ROC-AUC?". A model that predicts "up" on nearly every day will match the always-up baseline and nothing more.

In [ ]:
# ============================================================
#  CELL 10: Model comparison against the baselines
# ============================================================
ensemble_prob = (rf_prob + xgb_prob + bilstm_prob) / 3
model_probs = {'Random Forest': rf_prob, 'XGBoost': xgb_prob, 'BiLSTM': bilstm_prob,
               'Ensemble (average)': ensemble_prob}

THRESHOLD['Ensemble (average)'] = float(np.median(ensemble_prob[VAL]))
for k in baseline_probs: THRESHOLD[k] = 0.5
val_table  = pd.DataFrame({k: evaluate(y[VAL], v[VAL], THRESHOLD[k]) for k, v in model_probs.items()}).T
BEST_MODEL = val_table['Balanced acc'].idxmax()
print(f"Best model on the validation block: {BEST_MODEL}\n")

test_table = pd.DataFrame({k: evaluate(y[TEST], v[TEST], THRESHOLD[k]) for k, v in {**model_probs, **baseline_probs}.items()}).T
print("TEST BLOCK (touched once)")
display(test_table.style.format('{:.3f}').highlight_max(subset=['Balanced acc', 'MCC', 'ROC-AUC'], color='#cde8f6'))

au = test_table.loc['Baseline: always up']
print(f"Always-up baseline on the test block: F1 {au['F1 (up)']:.3f}, accuracy {au['Accuracy']:.3f}")
for name in model_probs:
    beats = test_table.loc[name, 'Balanced acc'] > 0.5 and test_table.loc[name, 'MCC'] > 0
    print(f"  {name:<20} balanced acc {test_table.loc[name, 'Balanced acc']:.3f}  "
          f"MCC {test_table.loc[name, 'MCC']:+.3f}  {'above a coin flip here (see the significance tests)' if beats else 'no better than a coin flip'}")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for (name, prob), color in zip(model_probs.items(), ['#e74c3c', '#f39c12', '#2980b9', '#8e44ad']):
    fpr, tpr, _ = roc_curve(y[TEST], prob[TEST])
    axes[0].plot(fpr, tpr, lw=2, color=color, label=f'{name} (AUC {roc_auc_score(y[TEST], prob[TEST]):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
axes[0].set_title('ROC curves, test block', fontweight='bold')
axes[0].set_xlabel('False positive rate'); axes[0].set_ylabel('True positive rate'); axes[0].legend()
test_table[['Balanced acc', 'ROC-AUC']].plot(kind='barh', ax=axes[1], color=['#2980b9', '#e74c3c'], edgecolor='black')
axes[1].axvline(0.5, color='black', lw=1, ls='--')
axes[1].set_xlim(0.35, 0.65)
axes[1].legend(loc='lower right')
axes[1].set_title('Test block: 0.5 is a coin flip', fontweight='bold')
plt.tight_layout(); plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight'); plt.show()

## Backtest

Long when the model's probability is above its threshold, in cash otherwise. The signal is formed from day *t*'s close and the position is held from that close to the next one, so this assumes trading at the closing auction. Every change of position pays `COST_BPS`. Exposure (the share of days invested) is shown, because a strategy that is invested 95% of the time is simply buy-and-hold.

In [ ]:
# ============================================================
#  CELL 11: Backtest with transaction costs
# ============================================================
def backtest(position, returns, cost_bps=COST_BPS):
    position = np.asarray(position, dtype=float)
    trades = np.abs(np.diff(np.concatenate([[0.0], position])))          # entering the first day counts
    daily = pd.Series(position * returns - trades * cost_bps / 1e4, index=dates[TEST])
    equity = (1 + daily).cumprod()
    years = len(daily) / TRADING_DAYS
    excess = daily - RISK_FREE / TRADING_DAYS
    dd = equity / equity.cummax() - 1
    return {
        'Total return':  equity.iloc[-1] - 1,
        'Annual return': equity.iloc[-1] ** (1 / years) - 1,
        'Annual vol':    daily.std() * np.sqrt(TRADING_DAYS),
        'Sharpe':        excess.mean() / excess.std() * np.sqrt(TRADING_DAYS) if excess.std() > 0 else 0.0,
        'Max drawdown':  dd.min(),
        'Exposure':      position.mean(),
        'Trades':        int(trades.sum()),
        '_equity': equity, '_drawdown': dd, '_daily': daily,
    }

rets_test = r_next[TEST]
runs = {'Buy and hold': backtest(np.ones(len(TEST)), rets_test, cost_bps=0)}
for name, prob in model_probs.items():
    runs[name] = backtest((prob[TEST] > THRESHOLD[name]).astype(int), rets_test)

bt_table = pd.DataFrame({k: {m: v for m, v in r.items() if not m.startswith('_')} for k, r in runs.items()}).T
display(bt_table.style.format({'Total return': '{:.2%}', 'Annual return': '{:.2%}', 'Annual vol': '{:.2%}',
                               'Sharpe': '{:.3f}', 'Max drawdown': '{:.2%}', 'Exposure': '{:.0%}', 'Trades': '{:.0f}'}))

best, bnh = runs[BEST_MODEL], runs['Buy and hold']
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
axes[0].plot(bnh['_equity'] * INITIAL_CAPITAL, color='#7f8c8d', lw=2, label='Buy and hold')
axes[0].plot(best['_equity'] * INITIAL_CAPITAL, color='#2980b9', lw=2, label=f'{BEST_MODEL} strategy (after costs)')
axes[0].axhline(INITIAL_CAPITAL, color='black', lw=0.7, ls='--')
axes[0].set_title(f'{LABEL}: equity curve on the test block', fontweight='bold'); axes[0].legend()
axes[1].fill_between(best['_drawdown'].index, best['_drawdown'] * 100, 0, alpha=0.6, color='#e74c3c', label=BEST_MODEL)
axes[1].fill_between(bnh['_drawdown'].index, bnh['_drawdown'] * 100, 0, alpha=0.3, color='#7f8c8d', label='Buy and hold')
axes[1].set_ylabel('Drawdown (%)'); axes[1].legend()
plt.tight_layout(); plt.savefig('backtest_results.png', dpi=150, bbox_inches='tight'); plt.show()

## Is the edge real, or luck?

Two tests on the test block:

1. **Binomial test.** A guesser with no skill that says "up" on the same share of days as the model would be right a predictable fraction of the time. Is the model's hit rate higher than that?
2. **Permutation test.** Shuffle the best model's daily positions thousands of times. This keeps its exposure and number of invested days but destroys any timing. If the real Sharpe ratio is not in the top 5% of the shuffled ones, the timing adds nothing.

In [ ]:
# ============================================================
#  CELL 12: Statistical significance
# ============================================================
best_pos = (model_probs[BEST_MODEL][TEST] > THRESHOLD[BEST_MODEL]).astype(int)
hits = int((best_pos == y[TEST]).sum())
q, u = best_pos.mean(), y[TEST].mean()
null_rate = q * u + (1 - q) * (1 - u)        # hit rate of random guessing that says "up" as often as the model
binom = stats.binomtest(hits, len(TEST), p=null_rate, alternative='greater')
print(f"{BEST_MODEL}: {hits}/{len(TEST)} correct ({hits/len(TEST):.1%}) vs {null_rate:.1%} "
      f"for random guessing with the same share of 'up' calls")
print(f"  Binomial test p-value: {binom.pvalue:.3f}")

real_sharpe = runs[BEST_MODEL]['Sharpe']
perm_sharpes = np.array([backtest(rng.permutation(best_pos), rets_test)['Sharpe'] for _ in range(N_PERMUTATIONS)])
perm_p = (np.sum(perm_sharpes >= real_sharpe) + 1) / (N_PERMUTATIONS + 1)
print(f"  Permutation test: Sharpe {real_sharpe:.3f}, p-value {perm_p:.3f} "
      f"({'timing looks real' if perm_p < 0.05 else 'not distinguishable from random timing'})")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(perm_sharpes, bins=50, color='#95a5a6', alpha=0.8, label='Shuffled positions')
ax.axvline(real_sharpe, color='#2980b9', lw=2.5, label=f'{BEST_MODEL} ({real_sharpe:.2f})')
ax.axvline(runs['Buy and hold']['Sharpe'], color='black', lw=1.5, ls='--', label='Buy and hold')
ax.set_title('Permutation test: Sharpe of the real timing vs random timing', fontweight='bold'); ax.legend()
plt.tight_layout(); plt.savefig('permutation_test.png', dpi=150, bbox_inches='tight'); plt.show()

## Walk-forward evaluation

A single train/test split trains once and then predicts a whole year with a model that never sees recent data. Walk-forward is closer to real use: retrain the tree models every `WF_RETRAIN_DAYS` days on all data available up to that point, predict the next block, and repeat across the test period.

In [ ]:
# ============================================================
#  CELL 13: Walk-forward retraining (tree models)
# ============================================================
wf_prob = {'Random Forest (walk-forward)': np.full(n, np.nan), 'XGBoost (walk-forward)': np.full(n, np.nan)}
for start in range(n_dev, n, WF_RETRAIN_DAYS):
    block = np.arange(start, min(start + WF_RETRAIN_DAYS, n))
    past = np.arange(0, start)                                    # everything known before the block
    rf_wf = RandomForestClassifier(**rf_params).fit(X_raw[past], y[past])
    xgb_wf = XGBClassifier(n_estimators=XGB_TREES, **xgb_params).fit(X_raw[past], y[past])
    wf_prob['Random Forest (walk-forward)'][block] = rf_wf.predict_proba(X_raw[block])[:, 1]
    wf_prob['XGBoost (walk-forward)'][block] = xgb_wf.predict_proba(X_raw[block])[:, 1]
print(f"Retrained {len(range(n_dev, n, WF_RETRAIN_DAYS))} times across the test block")

wf_thr = {'Random Forest (walk-forward)': THRESHOLD['Random Forest'], 'XGBoost (walk-forward)': THRESHOLD['XGBoost']}
wf_table = pd.DataFrame({k: {**evaluate(y[TEST], v[TEST], wf_thr[k]),
                             'Sharpe (after costs)': backtest((v[TEST] > wf_thr[k]).astype(int), rets_test)['Sharpe']}
                         for k, v in wf_prob.items()}).T
display(wf_table.style.format('{:.3f}'))

In [ ]:
# ============================================================
#  CELL 14: Confusion matrices on the test block
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, (name, prob) in zip(axes, model_probs.items()):
    pred = (prob[TEST] > THRESHOLD[name]).astype(int)
    sns.heatmap(confusion_matrix(y[TEST], pred, labels=[0, 1]), annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred down', 'Pred up'], yticklabels=['True down', 'True up'], cbar=False)
    ax.set_title(f"{name}\nbalanced acc {balanced_accuracy_score(y[TEST], pred):.3f}", fontweight='bold')
plt.tight_layout(); plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
#  CELL 15: SHAP explainability (XGBoost)
# ============================================================
X_test_df = pd.DataFrame(X_raw[TEST], columns=FEATURE_COLS, index=dates[TEST])
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_df)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df, max_display=20, show=False)
plt.title('SHAP feature impact: XGBoost, test block', fontweight='bold')
plt.tight_layout(); plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight'); plt.show()

shap.plots.waterfall(shap.Explanation(values=shap_values[-1], base_values=explainer.expected_value,
                                      data=X_test_df.iloc[-1].values, feature_names=FEATURE_COLS),
                     max_display=15, show=False)
plt.title(f'Why XGBoost predicted what it did on {X_test_df.index[-1].date()}', fontweight='bold')
plt.tight_layout(); plt.savefig('shap_waterfall.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ============================================================
#  CELL 16: Next-day signal from the latest data
# ============================================================
live_raw = features_ok.loc[[live_date], FEATURE_COLS].values           # the latest day, outcome unknown
live_window = scaler.transform(features_ok[FEATURE_COLS].values[-SEQUENCE_LEN:])[None, ...]

live = {
    'Random Forest': float(rf_model.predict_proba(live_raw)[0, 1]),
    'XGBoost':       float(xgb_model.predict_proba(live_raw)[0, 1]),
    'BiLSTM':        float(bilstm.predict(live_window, verbose=0)[0, 0]),
}
live['Ensemble (average)'] = float(np.mean(list(live.values())))
typical_move = raw_df['Close'].pct_change().abs().tail(21).mean()

print(f"Signal for the session after {live_date.date()} ({LABEL}, last close {raw_df['Close'].iloc[-1]:.2f})")
print(f"{'Model':<22}{'Direction':<12}{'P(up)':<9}Threshold")
for name, p in live.items():
    print(f"{name:<22}{'UP' if p > THRESHOLD[name] else 'DOWN':<12}{p:<9.1%}{THRESHOLD[name]:.1%}")
print(f"\nTypical daily move over the last month: +/-{typical_move:.2%}")
print(f"Model chosen on validation: {BEST_MODEL}. Read the significance tests above before trusting any signal.")
print("Educational project only, not financial advice.")

In [ ]:
# ============================================================
#  CELL 17: Summary dashboard
# ============================================================
close_s = raw_df['Close'].loc[df.index]
bb_plot = ta.volatility.BollingerBands(raw_df['Close'], window=20, window_dev=2)
macd_plot = ta.trend.MACD(raw_df['Close'])

fig = plt.figure(figsize=(18, 14))
fig.suptitle(f'{LABEL}: direction prediction summary', fontsize=16, fontweight='bold', y=1.01)
ax1 = fig.add_subplot(3, 3, (1, 3))
ax1.plot(close_s.index, close_s, color='steelblue', lw=1.5, label='Close')
ax1.fill_between(df.index, bb_plot.bollinger_lband().loc[df.index], bb_plot.bollinger_hband().loc[df.index],
                 alpha=0.15, color='orange', label='Bollinger bands')
ax1.plot(df.index, raw_df['Close'].ewm(span=50, adjust=False).mean().loc[df.index], color='limegreen', lw=1, ls='--', label='EMA 50')
ax1.axvspan(dates[VAL[0]], dates[VAL[-1]], color='gold', alpha=0.12, label='Validation')
ax1.axvspan(dates[TEST[0]], dates[TEST[-1]], color='red', alpha=0.08, label='Test')
ax1.set_title('Price, bands and the three data blocks', fontweight='bold'); ax1.legend(fontsize=8)

ax2 = fig.add_subplot(3, 3, 4)
ax2.plot(df.index, df['rsi_14'], color='purple', lw=1)
ax2.axhline(70, color='red', ls='--', alpha=0.7); ax2.axhline(30, color='green', ls='--', alpha=0.7)
ax2.set_title('RSI (14)', fontweight='bold')
ax3 = fig.add_subplot(3, 3, 5)
md_ = macd_plot.macd_diff().loc[df.index]
ax3.bar(df.index, md_, color=np.where(md_ >= 0, 'green', 'red'), alpha=0.5, width=1)
ax3.set_title('MACD histogram', fontweight='bold')
ax4 = fig.add_subplot(3, 3, 6)
ax4.plot(df.index, df['atr_pct'] * 100, color='darkorange', lw=1)
ax4.set_title('ATR as % of price', fontweight='bold')

ax5 = fig.add_subplot(3, 3, 7)
test_table[['Balanced acc', 'ROC-AUC']].plot(kind='barh', ax=ax5, color=['#2980b9', '#e74c3c'], edgecolor='black')
ax5.axvline(0.5, color='black', lw=0.8, ls='--'); ax5.set_xlim(0.3, 0.7)
ax5.set_title('Test block: models vs baselines', fontweight='bold')
ax6 = fig.add_subplot(3, 3, (8, 9))
ax6.plot(runs['Buy and hold']['_equity'] * INITIAL_CAPITAL, color='gray', lw=2, label='Buy and hold')
ax6.plot(runs[BEST_MODEL]['_equity'] * INITIAL_CAPITAL, color='#2980b9', lw=2, label=f'{BEST_MODEL} (after costs)')
ax6.axhline(INITIAL_CAPITAL, color='black', lw=0.8, ls='--')
ax6.set_title('Equity curve, test block', fontweight='bold'); ax6.legend()
plt.tight_layout(); plt.savefig('final_dashboard.png', dpi=150, bbox_inches='tight'); plt.show()

print(f"Features      : {len(FEATURE_COLS)}")
print(f"Best model    : {BEST_MODEL} (chosen on validation)")
print(f"Test balanced accuracy : {test_table.loc[BEST_MODEL, 'Balanced acc']:.3f}  "
      f"(always-up baseline {test_table.loc['Baseline: always up', 'Balanced acc']:.3f})")
print(f"Sharpe after costs     : {runs[BEST_MODEL]['Sharpe']:.3f}  (buy and hold {runs['Buy and hold']['Sharpe']:.3f})")
print(f"Permutation p-value    : {perm_p:.3f}")